# Brain MRI classification: ViT and Swin Transformer
### A cell-by-cell Google Colab notebook

**Goal:** explore the supplied dataset, fine-tune ImageNet-pretrained **ViT-B/16** and
**Swin-T**, and compare their held-out test accuracy and other classification metrics.

**Start:** upload this `.ipynb` to [Google Colab](https://colab.research.google.com/),
choose **Runtime → Change runtime type → T4 GPU** (or another GPU), then run the cells
in order. Internet access is needed for the dataset and pretrained weights. A CPU
runtime is supported but full fine-tuning is slow.

**Included:** automatic download; corrupt-image and exact-duplicate checks; class,
resolution and intensity EDA; reproducible splits; shared preprocessing; full
fine-tuning with mixed precision, gradient accumulation, warmup, early stopping,
and checkpoint resume; learning curves; test accuracy with image-level Wilson
intervals; classification reports; confusion matrices; ROC curves; error examples;
single-image inference; and a downloadable results archive.

This file contains **no invented accuracy values**. Tables and plots are generated
from your training run. Actual performance and runtime depend on the data and hardware.

## Context and methods

- Keep the supplied `Testing` folder as the held-out source. Remove corrupt files,
  ambiguous exact copies with conflicting labels, and redundant copies. If the same
  pixels appear in Training and Testing, keep the Testing copy and discard the
  Training copy. Then split the remaining Training images 80:20, stratified by class.
- This creates a **cleaned image-level benchmark**, whose sample counts can differ
  from the original Kaggle split. Raw and cleaned manifests are exported.
- Both models use the same splits, 224 × 224 RGB inputs, ImageNet normalization,
  training budget and hyperparameters. Validation macro-F1 selects checkpoints;
  validation loss breaks ties. Testing is evaluated only after both models train.
- ViT-B/16 and Swin-T have different capacities. Report the parameter counts rather
  than presenting this as a comparison at equal model size.

### Key assumptions and limits

The folder names provide image labels. They do **not** provide verified patient IDs.
Exact pixel deduplication cannot detect all near duplicates or related slices from
one patient. Scores and confidence intervals therefore describe this image-level
benchmark, not patient-independent clinical performance. Do not use the test set
to tune settings. One seed is a starting experiment; repeated seeds and an external
patient-level test set are needed for stronger research claims.

## Setup

### Cell 1 — Install only missing helper packages
Use the matching PyTorch and Torchvision pair supplied by a fresh Colab runtime.
Avoid separately upgrading one of those two packages, which can break their operators.

In [ ]:
import os
import sys
import subprocess
import importlib.util

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
helper_packages = {
    "kagglehub": "kagglehub>=0.3.13,<1",
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "seaborn": "seaborn", "sklearn": "scikit-learn", "PIL": "Pillow",
    "tqdm": "tqdm",
}
missing = [package for module, package in helper_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    raise RuntimeError("Use a fresh Google Colab runtime, which includes torch and torchvision.")
print("Helper packages are ready. PyTorch/Torchvision will be checked in Cell 2.")

### Cell 2 — Imports and runtime check

In [ ]:
import os
import sys
import gc
import json
import math
import time
import random
import hashlib
import platform
import zipfile
import importlib.metadata
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    f1_score, classification_report, confusion_matrix, roc_curve, auc,
    roc_auc_score,
)
try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    import torchvision
    from torchvision import transforms
    from torchvision.transforms import InterpolationMode
    from torchvision.models import (
        vit_b_16, ViT_B_16_Weights, swin_t, Swin_T_Weights,
    )
except Exception as exc:
    raise RuntimeError(
        "PyTorch/Torchvision could not import. In Colab, disconnect and delete the "
        "runtime, reconnect to a fresh GPU runtime, then run from Cell 1. "
        "Do not install a mismatched torch/torchvision pair."
    ) from exc
import kagglehub

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300})
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    print("For the full experiment, select a GPU runtime before proceeding.")

### Cell 3 — Experiment settings
Defaults run up to **20 epochs per model**, with early stopping. The physical batch
is 8; four accumulated batches give an effective batch of 32, except the final
partial group. Both backbones are trainable from the first epoch.

`QUICK_CHECK=True` runs a small one-epoch pipeline check with pretrained weights;
its metrics are not full-dataset experiment results. Use a new `RUN_NAME` when
changing training settings. Set `SAVE_TO_DRIVE=True` before your first run if you
want checkpoints to survive a Colab runtime reset. Resuming requires the same
settings and the same cleaned splits.

In [ ]:
CFG = {
    "DATASET_HANDLE": "masoudnickparvar/brain-tumor-mri-dataset",
    # Optional: append /versions/<version> to pin a Kaggle dataset version.
    "DATA_DIR": "",       # optional folder containing Training/ and Testing/
    "ZIP_PATH": "",       # optional downloaded Kaggle ZIP, e.g. /content/archive.zip
    "SEED": 42,
    "VAL_FRACTION": 0.20,
    "IMAGE_SIZE": 224,
    "EPOCHS": 20,
    "BATCH_SIZE": 8,
    "ACCUM_STEPS": 4,
    "NUM_WORKERS": 2,
    "BACKBONE_LR": 3e-5,
    "HEAD_LR": 3e-4,
    "WEIGHT_DECAY": 0.01,
    "HEAD_DROPOUT": 0.20,
    "LABEL_SMOOTHING": 0.10,
    "WARMUP_EPOCHS": 2,
    "PATIENCE": 5,
    "GRAD_CLIP": 1.0,
    "USE_AMP": True,
    "QUICK_CHECK": False,
    "SAVE_TO_DRIVE": False,
    "RUN_NAME": "vit_swin_v1",
    "RESUME_TRAINING": True,
    "SAVE_LAST_CHECKPOINT": True,  # includes optimizer/RNG state; larger than best.pt
    "DOWNLOAD_RESULTS": True,
    "ZIP_INCLUDE_BEST_WEIGHTS": False,  # enable for an archive with the two best models
}
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
MODEL_KEYS = ["vit_b_16", "swin_t"]
MODEL_LABELS = {"vit_b_16": "ViT-B/16", "swin_t": "Swin-T"}
if CFG["QUICK_CHECK"]:
    CFG.update(EPOCHS=1, WARMUP_EPOCHS=0, NUM_WORKERS=0)
    CFG["RUN_NAME"] += "_quick_check"
assert 0 < CFG["VAL_FRACTION"] < 1
assert CFG["IMAGE_SIZE"] == 224, "These pretrained model settings use 224-pixel inputs."
assert CFG["EPOCHS"] >= 1 and CFG["BATCH_SIZE"] >= 1 and CFG["ACCUM_STEPS"] >= 1
assert CFG["PATIENCE"] >= 1 and 0 <= CFG["LABEL_SMOOTHING"] < 1
assert 0 <= CFG["HEAD_DROPOUT"] < 1 and CFG["NUM_WORKERS"] >= 0
assert CFG["BACKBONE_LR"] > 0 and CFG["HEAD_LR"] > 0 and CFG["WEIGHT_DECAY"] >= 0
assert Path(CFG["RUN_NAME"]).name == CFG["RUN_NAME"] and CFG["RUN_NAME"] not in ("", ".", "..")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(CFG["SEED"])
AMP_ENABLED = bool(CFG["USE_AMP"] and DEVICE.type == "cuda")
WORK_ROOT = Path("/content/brain_mri_transformers")
if CFG["SAVE_TO_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive/Brain_MRI_Transformers")
OUT_DIR = WORK_ROOT / CFG["RUN_NAME"]
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = OUT_DIR / "plots"
PLOT_DIR.mkdir(exist_ok=True)
print("Output directory:", OUT_DIR)
print("Effective batch:", CFG["BATCH_SIZE"] * CFG["ACCUM_STEPS"])
print("Mixed precision:", AMP_ENABLED)
print("Quick pipeline check:", CFG["QUICK_CHECK"])

def save_figure(fig, name):
    fig.savefig(PLOT_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)

## Data and exploratory analysis
### Cell 4 — Download and find the dataset folders
Public download uses `kagglehub`. You can instead upload the Kaggle ZIP to Colab's
Files panel and set `ZIP_PATH`, or point `DATA_DIR` to an extracted copy. If Kaggle
requires consent/authentication, sign in to Kaggle and use `kagglehub.login()` in
a separate cell, or use the ZIP route. Never paste an API token into this notebook.

In [ ]:
def extract_dataset_zip(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (root / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError("The archive contains a path outside its extraction directory.")
        archive.extractall(root)
    return root

def find_dataset_root(base):
    base = Path(base)
    if not base.is_dir():
        raise FileNotFoundError(f"Dataset directory does not exist: {base}")
    candidates = [base]
    candidates.extend(p.parent for p in base.rglob("*")
                      if p.is_dir() and p.name.lower() == "training")
    matches = []
    for candidate in sorted(set(candidates)):
        children = {p.name.lower(): p for p in candidate.iterdir() if p.is_dir()}
        if "training" in children and "testing" in children:
            matches.append((candidate, children["training"], children["testing"]))
    if len(matches) != 1:
        raise ValueError(f"Expected one Training/Testing pair under {base}; found {len(matches)}.")
    return matches[0]

if CFG["DATA_DIR"]:
    downloaded_path = Path(CFG["DATA_DIR"])
elif CFG["ZIP_PATH"]:
    downloaded_path = extract_dataset_zip(Path(CFG["ZIP_PATH"]), Path("/content/brain_mri_data"))
else:
    try:
        downloaded_path = Path(kagglehub.dataset_download(CFG["DATASET_HANDLE"]))
    except Exception as exc:
        raise RuntimeError(
            "Kaggle download did not complete. Check internet access or authenticate "
            "with kagglehub.login(). Alternatively upload the Kaggle ZIP, set ZIP_PATH "
            "in Cell 3, and rerun from Cell 3."
        ) from exc
DATA_ROOT, TRAIN_FOLDER, TEST_FOLDER = find_dataset_root(downloaded_path)
print("Dataset:", DATA_ROOT)
print("Training source:", TRAIN_FOLDER)
print("Testing source:", TEST_FOLDER)

### Cell 5 — Scan images, labels, corruption and exact copies
Images are decoded with EXIF orientation applied, then converted to RGB for both
hashing and training. Pixel hashes include shape and RGB pixel values, so copies
with different filenames or metadata can still be detected. Recompressed/resized
near copies are not necessarily detected. Statistics use a thumbnail to keep this
scan light. Original source files are left in place; manifests select usable files.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def canonical_label(folder_name):
    name = folder_name.lower().replace("-", "").replace("_", "").replace(" ", "")
    aliases = {"gliomatumor": "glioma", "meningiomatumor": "meningioma",
               "pituitarytumor": "pituitary", "notumor": "notumor"}
    return aliases.get(name, name)

def load_rgb(path):
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert("RGB")

def scan_dataset(train_folder, test_folder, data_root):
    records, failures = [], []
    for source_split, folder in [("Training", train_folder), ("Testing", test_folder)]:
        seen_classes = set()
        for class_dir in sorted(p for p in folder.iterdir() if p.is_dir() and not p.name.startswith(".")):
            class_name = canonical_label(class_dir.name)
            if class_name not in CLASS_TO_ID:
                raise ValueError(f"Unexpected class directory: {class_dir}")
            seen_classes.add(class_name)
            paths = sorted(p for p in class_dir.rglob("*")
                           if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
            for path in tqdm(paths, desc=f"Scan {source_split}/{class_name}", leave=False):
                try:
                    with Image.open(path) as opened:
                        original_mode = opened.mode
                        image = ImageOps.exif_transpose(opened).convert("RGB")
                        image.load()
                    array = np.asarray(image, dtype=np.uint8)
                    pixel_hash = hashlib.sha256(
                        str(array.shape).encode("ascii") + array.tobytes()
                    ).hexdigest()
                    gray = image.convert("L")
                    gray.thumbnail((128, 128))
                    intensity = np.asarray(gray, dtype=np.float32) / 255.0
                    records.append({
                        "path": str(path), "relative_path": str(path.relative_to(data_root)),
                        "source_split": source_split, "class_name": class_name,
                        "label": CLASS_TO_ID[class_name], "width": image.width,
                        "height": image.height, "original_mode": original_mode,
                        "file_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                        "pixel_sha256": pixel_hash, "mean_intensity": float(intensity.mean()),
                        "std_intensity": float(intensity.std()),
                        "black_fraction": float((intensity < 0.02).mean()),
                    })
                except (OSError, ValueError, Image.DecompressionBombError) as exc:
                    failures.append({"path": str(path), "source_split": source_split,
                                     "class_name": class_name, "error": str(exc)[:200]})
        if seen_classes != set(CLASS_NAMES):
            raise ValueError(f"Missing classes in {source_split}: {set(CLASS_NAMES) - seen_classes}")
    if not records:
        raise RuntimeError("No decodable images were found.")
    return pd.DataFrame(records), pd.DataFrame(
        failures, columns=["path", "source_split", "class_name", "error"])

raw_df, corrupt_df = scan_dataset(TRAIN_FOLDER, TEST_FOLDER, DATA_ROOT)
raw_df.to_csv(OUT_DIR / "raw_image_manifest.csv", index=False)
corrupt_df.to_csv(OUT_DIR / "corrupt_images.csv", index=False)
print("Decodable images:", len(raw_df), "| Corrupt/unreadable images:", len(corrupt_df))
display(raw_df.groupby(["source_split", "class_name"]).size().unstack(fill_value=0)
        .reindex(columns=CLASS_NAMES))
display(raw_df[["width", "height", "mean_intensity", "std_intensity", "black_fraction"]]
        .describe().round(3))
print("Original image modes:", raw_df["original_mode"].value_counts().to_dict())
if not corrupt_df.empty:
    display(corrupt_df.head(10))

### Cell 6 — Plot raw class balance, dimensions and intensity

In [ ]:
source_counts = (raw_df.groupby(["source_split", "class_name"]).size()
                 .unstack(fill_value=0).reindex(index=["Training", "Testing"], columns=CLASS_NAMES))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.7))
source_counts.T.plot(kind="bar", ax=axes[0], color=["#2563eb", "#f59e0b"], rot=20)
axes[0].set(title="Raw class counts", xlabel="Class", ylabel="Images")
axes[1].scatter(raw_df["width"], raw_df["height"], s=7, alpha=0.25, color="#2563eb")
axes[1].set(title="Original image dimensions", xlabel="Width (pixels)", ylabel="Height (pixels)")
# Inspect training-source intensity only; no test-derived preprocessing parameters.
sns.histplot(data=raw_df[raw_df["source_split"] == "Training"], x="mean_intensity",
             hue="class_name", hue_order=CLASS_NAMES, bins=35, element="step",
             stat="density", common_norm=False, ax=axes[2])
axes[2].set(title="Training image intensity", xlabel="Mean gray intensity (0–1)")
fig.tight_layout()
save_figure(fig, "eda_raw_counts_dimensions_intensity")

### Cell 7 — Display sample training images

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(12, 11))
for row, class_name in enumerate(CLASS_NAMES):
    subset = raw_df[(raw_df["source_split"] == "Training") & (raw_df["class_name"] == class_name)]
    samples = subset.sample(n=min(4, len(subset)), random_state=CFG["SEED"])
    for col, ax in enumerate(axes[row]):
        ax.axis("off")
        if col < len(samples):
            sample = samples.iloc[col]
            ax.imshow(load_rgb(sample["path"]))
            ax.set_title(f"{class_name}\n{sample['width']} × {sample['height']}", fontsize=10)
fig.suptitle("Training examples before preprocessing", fontsize=15)
fig.tight_layout()
save_figure(fig, "eda_training_examples")

### Cell 8 — Clean exact duplicates and create train/validation/test manifests
The duplicate policy is fixed before training. It may alter the supplied benchmark,
so the notebook exports removal reasons and final counts. Within Testing, retain
one image per pixel hash. Within Training, discard hashes present in Testing and
retain one copy of every remaining hash. Conflicting labels for identical pixels
are removed entirely rather than choosing a label.

In [ ]:
def clean_exact_duplicates(frame):
    duplicate_rows = frame[frame.duplicated("pixel_sha256", keep=False)].copy()
    conflicts = frame.groupby("pixel_sha256")["class_name"].nunique()
    conflicting_hashes = set(conflicts[conflicts > 1].index)
    candidate = frame[~frame["pixel_sha256"].isin(conflicting_hashes)].copy()
    test_source = (candidate[candidate["source_split"] == "Testing"]
                   .sort_values("relative_path").drop_duplicates("pixel_sha256", keep="first"))
    test_hashes = set(test_source["pixel_sha256"])
    train_source = candidate[(candidate["source_split"] == "Training") &
                             ~candidate["pixel_sha256"].isin(test_hashes)]
    train_source = train_source.sort_values("relative_path").drop_duplicates("pixel_sha256", keep="first")
    retained_paths = set(train_source["path"]) | set(test_source["path"])
    removed = frame[~frame["path"].isin(retained_paths)].copy()
    def removal_reason(row):
        if row["pixel_sha256"] in conflicting_hashes:
            return "conflicting_labels_for_identical_pixels"
        if row["source_split"] == "Training" and row["pixel_sha256"] in test_hashes:
            return "training_copy_of_testing_image"
        return "redundant_copy_within_source_split"
    removed["removal_reason"] = removed.apply(removal_reason, axis=1) if len(removed) else pd.Series(dtype=str)
    return train_source.reset_index(drop=True), test_source.reset_index(drop=True), removed, duplicate_rows

train_source_df, test_df, removed_df, duplicate_df = clean_exact_duplicates(raw_df)
removed_df.to_csv(OUT_DIR / "removed_exact_duplicates.csv", index=False)
duplicate_df.to_csv(OUT_DIR / "exact_duplicate_groups.csv", index=False)
print("Rows in exact-copy groups:", len(duplicate_df), "| Removed rows:", len(removed_df))
if not removed_df.empty:
    display(removed_df["removal_reason"].value_counts().rename("images").to_frame())
for split_name, frame in [("Training source", train_source_df), ("Testing", test_df)]:
    if set(frame["class_name"]) != set(CLASS_NAMES):
        raise ValueError(f"Cleaning removed every usable image of a class in {split_name}.")
if train_source_df.groupby("class_name").size().min() < 2:
    raise ValueError("Each class needs at least two distinct training-source images for stratification.")
train_df, val_df = train_test_split(
    train_source_df, test_size=CFG["VAL_FRACTION"], random_state=CFG["SEED"],
    stratify=train_source_df["label"],
)
train_df, val_df, test_df = [frame.reset_index(drop=True) for frame in (train_df, val_df, test_df)]
if CFG["QUICK_CHECK"]:
    def cap_per_class(frame, limit):
        return pd.concat([group.sample(n=min(limit, len(group)), random_state=CFG["SEED"])
                          for _, group in frame.groupby("label")], ignore_index=True)
    train_df, val_df, test_df = (cap_per_class(train_df, 8), cap_per_class(val_df, 4),
                               cap_per_class(test_df, 4))
split_frames = {"train": train_df, "validation": val_df, "test": test_df}
for split_name, frame in split_frames.items():
    assert len(frame) and set(frame["label"]) == set(range(len(CLASS_NAMES))), split_name
    assert frame["pixel_sha256"].is_unique
    frame.to_csv(OUT_DIR / f"{split_name}_manifest.csv", index=False)
for first, second in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert set(split_frames[first]["pixel_sha256"]).isdisjoint(split_frames[second]["pixel_sha256"])
split_manifest = pd.concat([frame.assign(split=name) for name, frame in split_frames.items()], ignore_index=True)
split_manifest.to_csv(OUT_DIR / "split_manifest.csv", index=False)
fingerprint_lines = sorted(f"{row.split}|{row.class_name}|{row.pixel_sha256}"
                           for row in split_manifest.itertuples())
SPLIT_FINGERPRINT = hashlib.sha256("\n".join(fingerprint_lines).encode()).hexdigest()
split_counts = (split_manifest.groupby(["split", "class_name"]).size().unstack(fill_value=0)
                .reindex(index=["train", "validation", "test"], columns=CLASS_NAMES))
display(split_counts.assign(total=split_counts.sum(axis=1)))
fig, ax = plt.subplots(figsize=(9, 4.5))
split_counts.T.plot(kind="bar", ax=ax, rot=20, color=["#2563eb", "#14b8a6", "#f59e0b"])
ax.set(title="Cleaned experiment split counts", xlabel="Class", ylabel="Images")
fig.tight_layout()
save_figure(fig, "eda_cleaned_splits")
print("Exact-copy separation checks passed. Split fingerprint:", SPLIT_FINGERPRINT[:16])

## Preprocessing and model training
### Cell 9 — Shared preprocessing and data loaders
Letterboxing keeps the image aspect ratio and field of view, with black padding
where needed. Both architectures receive the same transform. Training uses mild
rotation/translation/scale augmentation; validation and Testing are deterministic.
There is no vertical flipping, aggressive crop, color jitter or fitted test-set
normalization. RGB conversion also handles grayscale inputs.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class LetterboxResize:
    def __init__(self, size=224):
        self.size = int(size)
    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), method=Image.Resampling.BICUBIC,
                            color=(0, 0, 0), centering=(0.5, 0.5))

train_transform = transforms.Compose([
    LetterboxResize(CFG["IMAGE_SIZE"]),
    transforms.RandomAffine(degrees=10, translate=(0.03, 0.03), scale=(0.95, 1.05),
                            interpolation=InterpolationMode.BILINEAR, fill=0),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    LetterboxResize(CFG["IMAGE_SIZE"]),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class MRIDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, index):
        return self.transform(load_rgb(self.paths[index])), self.labels[index]

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loaders(seed):
    train_generator = torch.Generator().manual_seed(seed)
    eval_generator = torch.Generator().manual_seed(seed + 1)
    workers = CFG["NUM_WORKERS"] if os.name == "posix" else 0
    common = dict(batch_size=CFG["BATCH_SIZE"], num_workers=workers,
                  pin_memory=DEVICE.type == "cuda", worker_init_fn=seed_worker,
                  persistent_workers=False, drop_last=False)
    loaders = {
        "train": DataLoader(MRIDataset(train_df, train_transform), shuffle=True,
                            generator=train_generator, **common),
        "validation": DataLoader(MRIDataset(val_df, eval_transform), shuffle=False,
                                 generator=eval_generator, **common),
    }
    return loaders, train_generator

def make_test_loader():
    return DataLoader(MRIDataset(test_df, eval_transform), batch_size=CFG["BATCH_SIZE"],
                      shuffle=False, num_workers=CFG["NUM_WORKERS"] if os.name == "posix" else 0,
                      pin_memory=DEVICE.type == "cuda", worker_init_fn=seed_worker,
                      generator=torch.Generator().manual_seed(CFG["SEED"] + 2), drop_last=False)

sample_tensor, sample_label = MRIDataset(train_df, eval_transform)[0]
assert sample_tensor.shape == (3, CFG["IMAGE_SIZE"], CFG["IMAGE_SIZE"])
assert torch.isfinite(sample_tensor).all()
print("Input shape:", tuple(sample_tensor.shape), "| Example class:", CLASS_NAMES[sample_label])

### Cell 10 — Inspect the shared preprocessing

In [ ]:
example_path = train_df.iloc[0]["path"]
example_image = load_rgb(example_path)
augmented = train_transform(example_image)
mean_tensor = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std_tensor = torch.tensor(IMAGENET_STD).view(3, 1, 1)
augmented_display = (augmented * std_tensor + mean_tensor).clamp(0, 1).permute(1, 2, 0).numpy()
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
axes[0].imshow(example_image); axes[0].set_title("Original")
axes[1].imshow(LetterboxResize(CFG["IMAGE_SIZE"])(example_image)); axes[1].set_title("Letterboxed evaluation")
axes[2].imshow(augmented_display); axes[2].set_title("Training augmentation")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
save_figure(fig, "preprocessing_examples")

### Cell 11 — Model builders, optimizer and learning-rate schedule
The first download is approximately 330 MB for ViT and 108 MB for Swin. The
1000-class ImageNet heads are replaced with dropout plus a four-class linear head.
Both backbones are fully fine-tuned with a lower learning rate than the new head.
AdamW does not apply weight decay to biases and one-dimensional parameters such
as normalization scales. These are starting settings, not a claim of optimality.

In [ ]:
WEIGHT_NAMES = {"vit_b_16": "ViT_B_16_Weights.IMAGENET1K_V1",
                "swin_t": "Swin_T_Weights.IMAGENET1K_V1"}

def build_model(model_key, pretrained=True):
    try:
        if model_key == "vit_b_16":
            model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None)
            input_features = model.heads.head.in_features
            model.heads = nn.Sequential(nn.Dropout(CFG["HEAD_DROPOUT"]),
                                        nn.Linear(input_features, len(CLASS_NAMES)))
            classifier = model.heads[-1]
        elif model_key == "swin_t":
            model = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1 if pretrained else None)
            input_features = model.head.in_features
            model.head = nn.Sequential(nn.Dropout(CFG["HEAD_DROPOUT"]),
                                      nn.Linear(input_features, len(CLASS_NAMES)))
            classifier = model.head[-1]
        else:
            raise ValueError(f"Unknown model: {model_key}")
    except (OSError, RuntimeError) as exc:
        raise RuntimeError(
            f"Could not create {model_key}. Check network access for pretrained weights "
            "and that PyTorch/Torchvision are a compatible pair."
        ) from exc
    nn.init.trunc_normal_(classifier.weight, std=0.02)
    nn.init.zeros_(classifier.bias)
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    return model

def is_head_parameter(model_key, name):
    return name.startswith("heads.") if model_key == "vit_b_16" else name.startswith("head.")

def make_optimizer(model, model_key):
    groups = {}
    for name, parameter in model.named_parameters():
        head = is_head_parameter(model_key, name)
        decay = parameter.ndim > 1 and not name.endswith(".bias")
        groups.setdefault((head, decay), []).append(parameter)
    parameter_groups = [
        {"params": parameters, "lr": CFG["HEAD_LR"] if head else CFG["BACKBONE_LR"],
         "weight_decay": CFG["WEIGHT_DECAY"] if decay else 0.0}
        for (head, decay), parameters in groups.items()
    ]
    return torch.optim.AdamW(parameter_groups)

def make_scheduler(optimizer):
    warmup = min(CFG["WARMUP_EPOCHS"], max(0, CFG["EPOCHS"] - 1))
    def factor(epoch_index):
        if warmup and epoch_index < warmup:
            return (epoch_index + 1) / warmup
        progress = np.clip((epoch_index - warmup) / max(1, CFG["EPOCHS"] - warmup - 1), 0, 1)
        return float(0.10 + 0.90 * 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=factor)

def make_scaler():
    # Modern API, with a compatibility branch for older Colab torch releases.
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

def amp_context():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if AMP_ENABLED else nullcontext()

def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters())

### Cell 12 — Training and validation loops
Gradient accumulation divides summed per-image losses by the number of images in
the current accumulation group, including the final short batch. This avoids
underweighting a partial group. All accuracy and F1 values use every processed image.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_images = 0.0, 0
    all_true, all_pred = [], []
    if training:
        optimizer.zero_grad(set_to_none=True)
    progress = tqdm(loader, desc="Train" if training else "Validation", leave=False)
    with torch.set_grad_enabled(training):
        for batch_index, (images, targets) in enumerate(progress):
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            with amp_context():
                logits = model(images)
                losses = criterion(logits, targets)
                loss_sum = losses.sum()
            if not torch.isfinite(loss_sum):
                raise FloatingPointError("Non-finite loss; check inputs and lower the learning rates.")
            if training:
                if batch_index % CFG["ACCUM_STEPS"] == 0:
                    group_batches = min(CFG["ACCUM_STEPS"], len(loader) - batch_index)
                    group_images = min(group_batches * loader.batch_size,
                                       len(loader.dataset) - batch_index * loader.batch_size)
                scaler.scale(loss_sum / group_images).backward()
                group_end = ((batch_index + 1) % CFG["ACCUM_STEPS"] == 0 or
                             batch_index + 1 == len(loader))
                if group_end:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            batch_images = targets.size(0)
            total_images += batch_images
            total_loss += float(loss_sum.detach())
            all_true.extend(targets.detach().cpu().tolist())
            all_pred.extend(logits.detach().argmax(dim=1).cpu().tolist())
            if batch_index % 20 == 0:
                progress.set_postfix(loss=f"{total_loss / total_images:.4f}")
    assert total_images == len(loader.dataset)
    return {
        "loss": total_loss / total_images,
        "accuracy": float(accuracy_score(all_true, all_pred)),
        "macro_f1": float(f1_score(all_true, all_pred, labels=range(len(CLASS_NAMES)),
                                    average="macro", zero_division=0)),
    }

@torch.inference_mode()
def predict_loader(model, loader):
    model.eval()
    truths, probabilities = [], []
    total_nll, count = 0.0, 0
    for images, targets in tqdm(loader, desc="Held-out evaluation", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        with amp_context():
            logits = model(images)
        # Float32 probabilities / NLL outside autocast for stable metrics.
        logits = logits.float()
        total_nll += float(nn.functional.cross_entropy(logits, targets, reduction="sum"))
        count += len(targets)
        truths.append(targets.cpu().numpy())
        probabilities.append(logits.softmax(dim=1).cpu().numpy())
    y_true = np.concatenate(truths)
    probs = np.concatenate(probabilities)
    assert count == len(loader.dataset) and np.isfinite(probs).all()
    assert np.allclose(probs.sum(axis=1), 1.0, atol=1e-5)
    return y_true, probs, total_nll / count

### Cell 13 — Portable checkpoints and resumable experiment runner
`best.pt` contains the validation-selected weights; `last.pt` also stores optimizer,
scheduler, mixed-precision and RNG states. Last checkpoints can be large (especially
for ViT). Exact-copy split fingerprints and training settings protect against loading
a checkpoint from another experiment. If memory is insufficient, reduce
`BATCH_SIZE` to 4 and increase `ACCUM_STEPS` to 8, choose a new `RUN_NAME`, and rerun
from Cell 3. If worker processes fail, set `NUM_WORKERS=0` and use a new `RUN_NAME`.

In [ ]:
SIGNATURE_KEYS = ["SEED", "IMAGE_SIZE", "EPOCHS", "BATCH_SIZE", "ACCUM_STEPS",
                  "NUM_WORKERS", "BACKBONE_LR", "HEAD_LR", "WEIGHT_DECAY",
                  "HEAD_DROPOUT", "LABEL_SMOOTHING", "WARMUP_EPOCHS", "PATIENCE",
                  "GRAD_CLIP", "USE_AMP", "QUICK_CHECK"]

def experiment_signature(model_key):
    payload = {key: CFG[key] for key in SIGNATURE_KEYS}
    payload.update(model_key=model_key, weight_name=WEIGHT_NAMES[model_key],
                   classes=CLASS_NAMES, split_fingerprint=SPLIT_FINGERPRINT,
                   preprocessing="rgb_letterbox_imagenet_affine10_translate0.03_scale0.95_1.05_v1")
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def atomic_torch_save(payload, path):
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)

def load_checkpoint(path):
    # Own checkpoints use only tensors, lists and primitive Python data.
    return torch.load(path, map_location="cpu", weights_only=True)

def capture_rng(train_generator):
    state = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": {"name": state[0], "keys": state[1].tolist(), "position": int(state[2]),
                  "has_gauss": int(state[3]), "cached_gaussian": float(state[4])},
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else [],
        "train_generator": train_generator.get_state(),
    }

def restore_rng(state, train_generator):
    random.setstate(state["python"])
    value = state["numpy"]
    np.random.set_state((value["name"], np.array(value["keys"], dtype=np.uint32),
                         value["position"], value["has_gauss"], value["cached_gaussian"]))
    torch.set_rng_state(state["torch_cpu"])
    if DEVICE.type == "cuda" and state["torch_cuda"]:
        torch.cuda.set_rng_state_all(state["torch_cuda"])
    train_generator.set_state(state["train_generator"])

def fit_model(model_key):
    seed_everything(CFG["SEED"])
    model_dir = OUT_DIR / model_key
    model_dir.mkdir(exist_ok=True)
    best_path, last_path = model_dir / "best.pt", model_dir / "last.pt"
    signature = experiment_signature(model_key)
    checkpoint = None
    if CFG["RESUME_TRAINING"] and last_path.exists():
        checkpoint = load_checkpoint(last_path)
        if checkpoint["signature"] != signature:
            raise ValueError("Existing checkpoint uses different settings/splits. Choose a new RUN_NAME.")
        if not best_path.exists():
            raise FileNotFoundError("Resume needs both last.pt and best.pt; choose a new RUN_NAME.")
    elif best_path.exists():
        raise FileExistsError("Existing best.pt found without a resumable last checkpoint. Choose a new RUN_NAME.")
    model = build_model(model_key, pretrained=checkpoint is None).to(DEVICE)
    params = parameter_count(model)
    optimizer = make_optimizer(model, model_key)
    scheduler = make_scheduler(optimizer)
    scaler = make_scaler()
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["LABEL_SMOOTHING"], reduction="none")
    loaders, train_generator = make_loaders(CFG["SEED"])
    history, start_epoch, bad_epochs = [], 0, 0
    best_f1, best_loss, best_epoch = -1.0, float("inf"), 0
    elapsed_training_seconds, finished = 0.0, False
    if checkpoint is not None:
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        scaler.load_state_dict(checkpoint["scaler_state"])
        history = checkpoint["history"]
        start_epoch = checkpoint["epoch"]
        best_f1, best_loss = checkpoint["best_f1"], checkpoint["best_loss"]
        best_epoch, bad_epochs = checkpoint["best_epoch"], checkpoint["bad_epochs"]
        elapsed_training_seconds = checkpoint["training_seconds"]
        finished = checkpoint["finished"]
        restore_rng(checkpoint["rng"], train_generator)
        print(f"Resuming {MODEL_LABELS[model_key]} after epoch {start_epoch}; complete={finished}")
        del checkpoint
        gc.collect()
    print(f"{MODEL_LABELS[model_key]}: {params / 1e6:.2f} million trainable parameters")
    if finished:
        epoch_range = []
    else:
        epoch_range = range(start_epoch, CFG["EPOCHS"])
    for epoch_index in epoch_range:
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        epoch_started = time.perf_counter()
        current_lrs = [group["lr"] for group in optimizer.param_groups]
        train_metrics = run_epoch(model, loaders["train"], criterion, optimizer, scaler)
        val_metrics = run_epoch(model, loaders["validation"], criterion)
        scheduler.step()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        epoch_seconds = time.perf_counter() - epoch_started
        elapsed_training_seconds += epoch_seconds
        history.append({"epoch": epoch_index + 1, "train_loss": train_metrics["loss"],
                        "train_accuracy": train_metrics["accuracy"], "train_macro_f1": train_metrics["macro_f1"],
                        "val_loss": val_metrics["loss"], "val_accuracy": val_metrics["accuracy"],
                        "val_macro_f1": val_metrics["macro_f1"], "epoch_seconds": epoch_seconds,
                        "min_lr": min(current_lrs), "max_lr": max(current_lrs)})
        improved = (val_metrics["macro_f1"] > best_f1 + 1e-12 or
                    (abs(val_metrics["macro_f1"] - best_f1) <= 1e-12 and val_metrics["loss"] < best_loss))
        if improved:
            best_f1, best_loss = val_metrics["macro_f1"], val_metrics["loss"]
            best_epoch, bad_epochs = epoch_index + 1, 0
            atomic_torch_save({
                "model_key": model_key, "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "class_names": CLASS_NAMES, "head_dropout": CFG["HEAD_DROPOUT"],
                "image_size": CFG["IMAGE_SIZE"], "signature": signature,
                "split_fingerprint": SPLIT_FINGERPRINT, "epoch": best_epoch,
                "val_macro_f1": best_f1, "val_loss": best_loss,
                "val_accuracy": val_metrics["accuracy"], "weights": WEIGHT_NAMES[model_key],
            }, best_path)
        else:
            bad_epochs += 1
        finished = bad_epochs >= CFG["PATIENCE"] or epoch_index + 1 >= CFG["EPOCHS"]
        pd.DataFrame(history).to_csv(model_dir / "training_history.csv", index=False)
        if CFG["SAVE_LAST_CHECKPOINT"]:
            atomic_torch_save({
                "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(), "rng": capture_rng(train_generator),
                "epoch": epoch_index + 1, "history": history, "signature": signature,
                "best_f1": best_f1, "best_loss": best_loss, "best_epoch": best_epoch,
                "bad_epochs": bad_epochs, "finished": finished,
                "training_seconds": elapsed_training_seconds,
            }, last_path)
        print(f"Epoch {epoch_index + 1:02d}/{CFG['EPOCHS']} | "
              f"train acc={100 * train_metrics['accuracy']:.2f}% | "
              f"val acc={100 * val_metrics['accuracy']:.2f}% | "
              f"val macro-F1={val_metrics['macro_f1']:.4f} | "
              f"best epoch={best_epoch} | {epoch_seconds:.1f}s")
        if finished:
            if bad_epochs >= CFG["PATIENCE"]:
                print("Early stopping: validation checkpoint criterion stopped improving.")
            break
    if not best_path.exists():
        raise RuntimeError("Training did not produce a finite best checkpoint.")
    record = {"model_key": model_key, "best_path": str(best_path), "params": params,
              "best_epoch": best_epoch, "best_val_macro_f1": best_f1,
              "training_seconds": elapsed_training_seconds, "epochs_run": len(history)}
    (model_dir / "training_summary.json").write_text(json.dumps(record, indent=2))
    del model, optimizer, scheduler, scaler, criterion, loaders
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return record

### Cell 14 — Train ViT-B/16
The test loader is not used here. Do not change hyperparameters after viewing test
results if you want to keep this test set held out.

In [ ]:
training_records = {}
training_records["vit_b_16"] = fit_model("vit_b_16")

### Cell 15 — Train Swin-T with the same protocol

In [ ]:
training_records["swin_t"] = fit_model("swin_t")
display(pd.DataFrame(training_records.values())[["model_key", "params", "best_epoch",
                                                "best_val_macro_f1", "epochs_run", "training_seconds"]])

### Cell 16 — Plot training/validation accuracy and loss
Training accuracy uses augmented images during optimization; validation accuracy
uses deterministic images. Values can therefore differ even with no overfitting.
The dashed line marks each model's validation-selected checkpoint.

In [ ]:
histories = {key: pd.read_csv(OUT_DIR / key / "training_history.csv") for key in MODEL_KEYS}
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
for row, key in enumerate(MODEL_KEYS):
    history = histories[key]
    axes[row, 0].plot(history["epoch"], 100 * history["train_accuracy"], label="Train", color="#2563eb")
    axes[row, 0].plot(history["epoch"], 100 * history["val_accuracy"], label="Validation", color="#f59e0b")
    axes[row, 0].set(title=f"{MODEL_LABELS[key]} — accuracy", xlabel="Epoch", ylabel="Accuracy (%)", ylim=(0, 100))
    axes[row, 1].plot(history["epoch"], history["train_loss"], label="Train", color="#2563eb")
    axes[row, 1].plot(history["epoch"], history["val_loss"], label="Validation", color="#f59e0b")
    axes[row, 1].set(title=f"{MODEL_LABELS[key]} — smoothed cross-entropy", xlabel="Epoch", ylabel="Loss")
    for ax in axes[row]:
        ax.axvline(training_records[key]["best_epoch"], color="#16a34a", linestyle="--", label="Selected epoch")
        ax.legend()
fig.tight_layout()
save_figure(fig, "training_accuracy_loss")
fig, ax = plt.subplots(figsize=(8, 4.5))
for key in MODEL_KEYS:
    ax.plot(histories[key]["epoch"], histories[key]["val_macro_f1"], marker="o", ms=3, label=MODEL_LABELS[key])
ax.set(title="Validation macro-F1 by epoch", xlabel="Epoch", ylabel="Macro-F1", ylim=(0, 1))
ax.legend()
fig.tight_layout()
save_figure(fig, "validation_macro_f1")

## Held-out results
### Cell 17 — Evaluate each best checkpoint on the cleaned Testing set
No model is trained in this cell. It loads the validation-selected checkpoint for
each architecture and evaluates every cleaned test image in manifest order. Test
loss is ordinary cross-entropy (NLL), whereas training/validation losses above use
label smoothing. The majority-class accuracy is a descriptive reference only.

In [ ]:
def wilson_interval(correct, total, z=1.959963984540054):
    if total <= 0 or not 0 <= correct <= total:
        raise ValueError("Wilson interval requires total > 0 and 0 <= correct <= total.")
    proportion = correct / total
    denominator = 1.0 + z * z / total
    center = (proportion + z * z / (2.0 * total)) / denominator
    half_width = z * math.sqrt(proportion * (1 - proportion) / total + z * z / (4 * total**2)) / denominator
    lower = 0.0 if correct == 0 else max(0.0, center - half_width)
    upper = 1.0 if correct == total else min(1.0, center + half_width)
    return lower, upper

def summarize_predictions(y_true, probs):
    y_pred = probs.argmax(axis=1)
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(CLASS_NAMES)), average="macro", zero_division=0)
    accuracy = float(accuracy_score(y_true, y_pred))
    ci_low, ci_high = wilson_interval(int((y_true == y_pred).sum()), len(y_true))
    return {
        "test_accuracy": accuracy,
        "accuracy_ci95_low": ci_low, "accuracy_ci95_high": ci_high,
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision), "macro_recall": float(recall),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_ovr_auc": float(roc_auc_score(y_true, probs, multi_class="ovr", average="macro",
                                             labels=np.arange(len(CLASS_NAMES)))),
    }

test_predictions, result_rows = {}, []
test_loader = make_test_loader()
for key in MODEL_KEYS:
    checkpoint = load_checkpoint(OUT_DIR / key / "best.pt")
    assert checkpoint["signature"] == experiment_signature(key)
    assert checkpoint["class_names"] == CLASS_NAMES
    model = build_model(key, pretrained=False).to(DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    y_true, probs, test_nll = predict_loader(model, test_loader)
    assert np.array_equal(y_true, test_df["label"].to_numpy()), "Test prediction order mismatch."
    y_pred = probs.argmax(axis=1)
    metrics = summarize_predictions(y_true, probs)
    record = training_records[key]
    row = {"model": MODEL_LABELS[key], "model_key": key, **metrics, "test_nll": test_nll,
           "test_images": len(y_true), "best_epoch": int(checkpoint["epoch"]),
           "selected_val_accuracy": float(checkpoint["val_accuracy"]),
           "selected_val_macro_f1": float(checkpoint["val_macro_f1"]),
           "trainable_params_millions": record["params"] / 1e6,
           "epochs_run": record["epochs_run"],
           "train_and_validation_minutes": record["training_seconds"] / 60.0,
           "quick_check": CFG["QUICK_CHECK"]}
    result_rows.append(row)
    test_predictions[key] = {"y_true": y_true, "y_pred": y_pred, "probs": probs}
    prediction_frame = test_df[["path", "relative_path", "class_name", "label"]].copy()
    prediction_frame["predicted_label"] = y_pred
    prediction_frame["predicted_class"] = [CLASS_NAMES[index] for index in y_pred]
    prediction_frame["confidence"] = probs.max(axis=1)
    prediction_frame["correct"] = y_true == y_pred
    for class_index, class_name in enumerate(CLASS_NAMES):
        prediction_frame[f"prob_{class_name}"] = probs[:, class_index]
    prediction_frame.to_csv(OUT_DIR / key / "test_predictions.csv", index=False)
    report = classification_report(y_true, y_pred, labels=range(len(CLASS_NAMES)),
                                   target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(OUT_DIR / key / "classification_report.csv")
    (OUT_DIR / key / "test_metrics.json").write_text(json.dumps(row, indent=2))
    matrix = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
    pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUT_DIR / key / "confusion_matrix.csv")
    del model, checkpoint
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
results_df = pd.DataFrame(result_rows)
results_df.to_csv(OUT_DIR / "model_comparison.csv", index=False)
display(results_df[["model", "test_accuracy", "accuracy_ci95_low", "accuracy_ci95_high",
                    "macro_f1", "macro_ovr_auc", "best_epoch", "trainable_params_millions"]]
        .style.format({"test_accuracy": "{:.2%}", "accuracy_ci95_low": "{:.2%}",
                       "accuracy_ci95_high": "{:.2%}", "macro_f1": "{:.4f}",
                       "macro_ovr_auc": "{:.4f}", "trainable_params_millions": "{:.2f}"}))
print("Test majority-class reference accuracy:", f"{test_df['label'].value_counts().max() / len(test_df):.2%}")

### Cell 18 — Plot the requested accuracy comparison
Bars show held-out test accuracy. Error bars are 95% Wilson intervals treating
images as independent; unknown patient relationships can make these intervals
too optimistic. They do not measure variation between repeated training runs.

In [ ]:
positions = np.arange(len(results_df))
test_percent = results_df["test_accuracy"].to_numpy() * 100
lower_error = test_percent - results_df["accuracy_ci95_low"].to_numpy() * 100
upper_error = results_df["accuracy_ci95_high"].to_numpy() * 100 - test_percent
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.1))
bars = axes[0].bar(positions, test_percent, color=["#2563eb", "#14b8a6"], width=0.55,
                   yerr=np.maximum(0, np.vstack([lower_error, upper_error])), capsize=7)
axes[0].set_xticks(positions, results_df["model"])
axes[0].set(title="Held-out test accuracy — " + ("QUICK CHECK" if CFG["QUICK_CHECK"] else "full experiment"),
            ylabel="Accuracy (%)", ylim=(0, 106))
axes[0].set_yticks(np.arange(0, 101, 20))
for bar, value, high in zip(bars, test_percent, results_df["accuracy_ci95_high"]):
    axes[0].text(bar.get_x() + bar.get_width() / 2, 100 * high + 1.5, f"{value:.2f}%",
                 ha="center", va="bottom", fontweight="bold")
width = 0.24
for offset, column, label, color in [(-width, "macro_precision", "Macro precision", "#2563eb"),
                                    (0, "macro_recall", "Macro recall", "#14b8a6"),
                                    (width, "macro_f1", "Macro F1", "#f59e0b")]:
    axes[1].bar(positions + offset, 100 * results_df[column], width=width, label=label, color=color)
axes[1].set_xticks(positions, results_df["model"])
axes[1].set(title="Test metrics across all four classes", ylabel="Score (%)", ylim=(0, 105))
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3, fontsize=8)
fig.tight_layout()
save_figure(fig, "test_accuracy_model_comparison")

### Cell 19 — Classification reports and confusion matrices

In [ ]:
for key in MODEL_KEYS:
    print("\n" + MODEL_LABELS[key])
    prediction = test_predictions[key]
    print(classification_report(prediction["y_true"], prediction["y_pred"],
                                labels=range(len(CLASS_NAMES)), target_names=CLASS_NAMES,
                                digits=4, zero_division=0))
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for row, key in enumerate(MODEL_KEYS):
    prediction = test_predictions[key]
    matrix = confusion_matrix(prediction["y_true"], prediction["y_pred"], labels=range(len(CLASS_NAMES)))
    normalized = matrix / np.maximum(1, matrix.sum(axis=1, keepdims=True))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[row, 0])
    sns.heatmap(normalized, annot=True, fmt=".1%", cmap="Blues", vmin=0, vmax=1, cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[row, 1])
    axes[row, 0].set_title(f"{MODEL_LABELS[key]} — counts")
    axes[row, 1].set_title(f"{MODEL_LABELS[key]} — row normalized")
    for ax in axes[row]:
        ax.set(xlabel="Predicted class", ylabel="True class")
        ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
save_figure(fig, "test_confusion_matrices")

### Cell 20 — One-versus-rest ROC curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, key in zip(axes, MODEL_KEYS):
    prediction = test_predictions[key]
    for class_index, class_name in enumerate(CLASS_NAMES):
        binary_truth = (prediction["y_true"] == class_index).astype(int)
        false_positive_rate, true_positive_rate, _ = roc_curve(binary_truth, prediction["probs"][:, class_index])
        class_auc = auc(false_positive_rate, true_positive_rate)
        ax.plot(false_positive_rate, true_positive_rate, label=f"{class_name}: AUC {class_auc:.3f}")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1)
    ax.set(title=f"{MODEL_LABELS[key]} — test ROC", xlabel="False-positive rate",
           ylabel="True-positive rate", xlim=(0, 1), ylim=(0, 1.03))
    ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
save_figure(fig, "test_roc_curves")

### Cell 21 — Inspect confident errors
These images help describe failure modes. Reviewing them is post-test analysis;
changing settings based on them would turn this test set into development data.
Softmax confidence is not a calibrated probability of a clinical diagnosis.

In [ ]:
for key in MODEL_KEYS:
    prediction = test_predictions[key]
    wrong_indices = np.flatnonzero(prediction["y_true"] != prediction["y_pred"])
    print(MODEL_LABELS[key], "misclassified images:", len(wrong_indices))
    if not len(wrong_indices):
        continue
    confidence = prediction["probs"].max(axis=1)
    selected = wrong_indices[np.argsort(-confidence[wrong_indices])][:8]
    fig, axes = plt.subplots(2, 4, figsize=(13, 6.7))
    for ax, image_index in zip(axes.flat, selected):
        ax.imshow(load_rgb(test_df.iloc[image_index]["path"]))
        actual = CLASS_NAMES[prediction["y_true"][image_index]]
        predicted = CLASS_NAMES[prediction["y_pred"][image_index]]
        ax.set_title(f"True: {actual}\nPred: {predicted} ({confidence[image_index]:.1%})", fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(f"{MODEL_LABELS[key]} — most confident test errors")
    fig.tight_layout()
    save_figure(fig, f"{key}_test_errors")

### Cell 22 — Optional single-image prediction
Set `CUSTOM_IMAGE_PATH` to an image uploaded to Colab's Files panel. The function
uses the saved best checkpoint and the same evaluation transform. Predictions are
benchmark class labels for research, not clinical advice.

In [ ]:
@torch.inference_mode()
def predict_image(image_path, model_key="swin_t"):
    if model_key not in MODEL_KEYS:
        raise ValueError(f"model_key must be one of {MODEL_KEYS}")
    image = load_rgb(image_path)
    checkpoint = load_checkpoint(OUT_DIR / model_key / "best.pt")
    assert checkpoint["class_names"] == CLASS_NAMES
    assert checkpoint["signature"] == experiment_signature(model_key)
    model = build_model(model_key, pretrained=False).to(DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    batch = eval_transform(image).unsqueeze(0).to(DEVICE)
    with amp_context():
        logits = model(batch)
    probabilities = logits.float().softmax(dim=1)[0].cpu().numpy()
    prediction_table = pd.DataFrame({"class": CLASS_NAMES, "probability": probabilities})
    prediction_table = prediction_table.sort_values("probability", ascending=False).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(image)
    ax.set_title(f"{MODEL_LABELS[model_key]}: {prediction_table.iloc[0]['class']}")
    ax.axis("off")
    plt.show()
    plt.close(fig)
    del model, checkpoint
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return prediction_table

CUSTOM_IMAGE_PATH = ""  # e.g. /content/my_scan.jpg
if CUSTOM_IMAGE_PATH:
    display(predict_image(CUSTOM_IMAGE_PATH, model_key="swin_t"))
else:
    print("Optional inference skipped. Set CUSTOM_IMAGE_PATH to try a saved model.")

## Checks and exports
### Cell 23 — Save metadata and download the results archive
The ZIP contains EDA plots, cleaned/raw manifests, duplicate audits, training logs,
predictions and comparison reports. By default it excludes large checkpoints.
Set `ZIP_INCLUDE_BEST_WEIGHTS=True` in Cell 3 to include both `best.pt` files;
optimizer resume files are always excluded. Training time excludes model downloads
and checkpoint writing. For a future rerun, use the exported package versions and
the recorded dataset path/version; the split fingerprint verifies image membership.

In [ ]:
versions = {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": str(torchvision.__version__)}
for package in ["kagglehub", "numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "Pillow", "tqdm"]:
    versions[package] = importlib.metadata.version(package)
metadata = {
    "config": CFG, "classes": CLASS_NAMES, "model_weights": WEIGHT_NAMES,
    "dataset_url": "https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset",
    "dataset_root": str(DATA_ROOT), "dataset_download_path": str(downloaded_path),
    "split_fingerprint": SPLIT_FINGERPRINT,
    "split_counts": {name: int(len(frame)) for name, frame in split_frames.items()},
    "raw_decodable_images": int(len(raw_df)), "corrupt_images": int(len(corrupt_df)),
    "removed_duplicate_or_conflicting_rows": int(len(removed_df)),
    "versions": versions, "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "evaluation_unit": "image; patient identities unavailable",
    "checkpoint_selection": "validation macro-F1, tie broken by validation loss",
    "preprocessing": "RGB, aspect-preserving black letterbox to 224, ImageNet mean/std",
    "data_limitations": "Exact duplicates checked; near duplicates and related patient slices may remain.",
}
(OUT_DIR / "experiment_metadata.json").write_text(json.dumps(metadata, indent=2))
requirements = "\n".join(f"{package}=={version}" for package, version in versions.items() if package != "python")
(OUT_DIR / "requirements_recorded.txt").write_text(requirements + "\n")
for key in MODEL_KEYS:
    assert len(histories[key]) >= 1
    assert len(test_predictions[key]["y_true"]) == len(test_df)
assert results_df["test_accuracy"].between(0, 1).all()
assert len(results_df) == 2 and results_df["model_key"].nunique() == 2
archive_path = WORK_ROOT / f"{CFG['RUN_NAME']}_results.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.rglob("*")):
        if not path.is_file() or path.suffix == ".tmp":
            continue
        if path.suffix == ".pt" and (path.name != "best.pt" or not CFG["ZIP_INCLUDE_BEST_WEIGHTS"]):
            continue
        archive.write(path, arcname=str(path.relative_to(OUT_DIR)))
print("Saved results:", archive_path)
print("Checkpoint directories:", *(str(OUT_DIR / key) for key in MODEL_KEYS), sep="\n")
if CFG["DOWNLOAD_RESULTS"]:
    try:
        from google.colab import files
    except ImportError:
        print("Outside Colab: open the ZIP directly from the path printed above.")
    else:
        files.download(str(archive_path))

## Next steps and interpretation

Read `model_comparison.csv` and `test_accuracy_model_comparison.png` for the
requested accuracy comparison. Check class-specific recall and error examples
alongside accuracy. Select model settings using validation data and report both
architectures on the same cleaned test set.

For a research experiment, turn off `QUICK_CHECK`, repeat fresh runs with several
seeds and new run names, report mean ± standard deviation, and use a verified
patient-level split or an external dataset where possible. These two baseline
models alone do not establish a novel or clinically validated method.

### Authoring validation status

The delivered notebook has blank outputs. All **23 Python cells** were syntax-checked.
The pipeline was exercised on synthetic RGB/grayscale images, including corrupt
files, exact copies, split overlap and label conflicts, with **one CPU epoch for
each real model architecture using randomly initialized weights**. Checks also
covered partial-batch gradient accumulation, checkpoint saving/loading and resume,
reports, plots, single-image inference, and result exports.

The supplied Kaggle dataset, pretrained downloads, CUDA mixed precision and full
MRI training were **not executed here**. Therefore **full pretrained execution remains
to be verified in a fresh Colab GPU runtime**. To reproduce that validation: choose a GPU,
keep `QUICK_CHECK=False`, run Cells 1–23 in order, and inspect the checks and exports
in Cell 23. For a short initial pipeline trial, set `QUICK_CHECK=True` first,
then use a new full-run name with `QUICK_CHECK=False`.

### Primary sources

- [The supplied Kaggle Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)
- [Official Kagglehub download/authentication documentation](https://github.com/Kaggle/kagglehub)
- [Torchvision ViT-B/16 and pretrained weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vit_b_16.html)
- [Torchvision Swin-T and pretrained weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.swin_t.html)
- [PyTorch AMP examples](https://docs.pytorch.org/docs/stable/notes/amp_examples.html)
- [ViT paper](https://arxiv.org/abs/2010.11929)
- [Swin Transformer paper](https://arxiv.org/abs/2103.14030)